In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from statsbombpy import sb

pd.set_option('display.max_columns', None)
pd.options.mode.chained_assignment = None  # default='warn'

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings(action="ignore", message="credentials were not supplied. open data access only")

In [11]:
competitions = sb.competitions()
selected_competitions = competitions[competitions.competition_name.isin(['Champions League', 'La Liga', 'Premier League', 'UEFA Euro'])]
selected_competitions = selected_competitions[selected_competitions.season_name != '1999/2000']

In [12]:
def get_match_ids_info(competitions):
  """
  Get the match IDs and information for each match in the selected competitions.

  Args:
    competitions: A pandas DataFrame of selected competitions.

  Returns:
    A tuple of two lists: `match_ids_list` and `match_ids_info`.
  """

  match_ids_list = []
  match_ids_info = {}

  for index, row in competitions.iterrows():
    matches = sb.matches(row.competition_id, row.season_id)

    for match in matches.itertuples():
      match_ids_list.append(match.match_id)
      match_ids_info[match.match_id] = [match.home_team, match.away_team]

  return match_ids_list, match_ids_info

In [13]:
def get_fouls(match_ids_list, columns, subset):
  """
  This method gets all the fouls from a list of match IDs.

  Args:
    match_ids_list: A list of match IDs.
    columns: A list of the columns that are interesting.
    subset: A list of the columns that should not be NaN.

  Returns:
    A DataFrame of all the fouls.
  """

  fouls = pd.DataFrame()

  for match_id in tqdm(match_ids_list):
    match_events_df = sb.events(match_id=match_id)
    match_events_df_fouls = match_events_df.reindex(columns=columns)

    match_events_df_fouls.dropna(subset=subset, how='all', inplace=True)

    fouls = pd.concat([fouls, match_events_df_fouls], ignore_index=True)

  return fouls

In [14]:
interesting_columns = ['foul_committed_advantage','foul_committed_offensive', 'foul_committed_penalty','foul_committed_card','foul_committed_counterpress','foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'foul_won_penalty', 'id', 'index', 'location', 'match_id', 'minute', 'second', 'period', 'player', 'player_id', 'position', 'possession', 'possession_team', 'possession_team_id', 'team', 'team_id', 'timestamp', 'type']
interesting_subset = ['foul_committed_advantage','foul_committed_card','foul_committed_offensive','foul_committed_penalty','foul_committed_counterpress','foul_committed_type']

match_ids_list, match_ids_info = get_match_ids_info(selected_competitions)
all_fouls = get_fouls(match_ids_list, interesting_columns, interesting_subset)

100%|██████████| 623/623 [07:57<00:00,  1.30it/s]


In [15]:
test_competitions = competitions[competitions.competition_name.isin(['FIFA World Cup'])]
match_ids_list_test, match_ids_info_test = get_match_ids_info(test_competitions)
testing_fouls = get_fouls(match_ids_list_test, interesting_columns, interesting_subset)

all_fouls.to_pickle('all_fouls.pkl')
testing_fouls.to_pickle('testing_fouls.pkl')

100%|██████████| 128/128 [01:36<00:00,  1.32it/s]


In [7]:
# all_fouls = pd.read_pickle('all_fouls.pkl')
# testing_fouls = pd.read_pickle('testing_fouls.pkl')

In [16]:
def get_goals(match_ids):
  """
  Get all goals from a list of match IDs.

  Args:
    match_ids: A list of match IDs.

  Returns:
    A DataFrame of goals and match events
  """

  goals = pd.DataFrame()
  match_events = pd.DataFrame()

  for match_id in tqdm(match_ids):
    match_events_df = sb.events(match_id=match_id)
    match_events = pd.concat([match_events, match_events_df], ignore_index=True)

    match_events_df_shots = match_events_df[match_events_df.type == 'Shot']
    match_events_df_goals = match_events_df_shots[match_events_df_shots.shot_outcome == 'Goal']

    match_events_df_goals['seconds'] = match_events_df_goals['minute'] * 60 + match_events_df_goals['second']
    goals = pd.concat([goals, match_events_df_goals], ignore_index=True)

  return goals, match_events

In [17]:
all_goals, match_events = get_goals(match_ids_list)
test_goals, match_events_test = get_goals(match_ids_list_test)

all_goals.to_pickle('all_goals.pkl')
test_goals.to_pickle('test_goals.pkl')

match_events.to_pickle('match_events.pkl')
match_events_test.to_pickle('match_events_test.pkl')

  6%|▋         | 8/128 [00:03<00:53,  2.23it/s]/tmp/ipykernel_38638/2781127628.py:23: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  goals = pd.concat([goals, match_events_df_goals], ignore_index=True)
100%|██████████| 128/128 [01:45<00:00,  1.21it/s]


In [10]:
# all_goals = pd.read_pickle('all_goals.pkl')
# test_goals = pd.read_pickle('test_goals.pkl')

# match_events = pd.read_pickle('match_events.pkl')
# match_events_test = pd.read_pickle('match_events_test.pkl')

In [18]:
def get_scoreline(goals, match_events, match_id, seconds):
    match_goals = goals[goals['match_id'] == match_id]
    
    # find unique team_id
    teams = match_events['team'].unique()
    match_goals = match_goals[match_goals['seconds'] <= seconds]

    # find goal scored by each_team
    team_goal_count = match_goals.groupby('team')['team'].count()
    team_goal_dict = {}
    
    for team in teams:
        # if team_goal_count['team'] is NaN, then add to team_goal_dict with key as team_name and value as 0
        if team not in team_goal_count:
            team_goal_dict[team] = 0
        else:
            team_goal_dict[team] = team_goal_count[team]
            
    return team_goal_dict


def distance_to_goal(isHomeTeam, location):
    if isHomeTeam:
        # calculate distance from location to away goal
        return np.sqrt((120 - location[0])**2 + (40 - location[1])**2)
    else:
        # calculate distance from location to home goal
        return np.sqrt((0 - location[0])**2 + (40 - location[1])**2)


def angle_to_goal(isHomeTeam, location):
    if isHomeTeam:
        # calculate angle from location to away goal
        return np.arctan((40 - location[1]) / (120 - location[0]))
    else:
        # calculate angle from location to home goal
        return np.arctan((40 - location[1]) / (0 - location[0]))

In [19]:
# Write method to calculate previous foul count of the player making the foul. Inputs will be match_id, player_id, timestamp
def previous_foul_count(fouls, match_id, player_id, seconds):
    # get all fouls in the match
    match_fouls_df = fouls[fouls['match_id'] == match_id]
    
    # get all fouls by player_id
    player_fouls_df = match_fouls_df[match_fouls_df['player_id'] == player_id]
    
    player_fouls_df['seconds'] = player_fouls_df['minute'] * 60 + player_fouls_df['second']

    # get all fouls before timestamp
    previous_fouls_df = player_fouls_df[player_fouls_df['seconds'] <= seconds]
    
    # return count of fouls
    return len(previous_fouls_df)


# Write method to calculate previous foul count of the team making the foul. Inputs will be match_id, team_id, timestamp
def previous_foul_count_team(fouls, match_id, team_id, seconds):
    # get all fouls in the match
    match_fouls_df = fouls[fouls['match_id'] == match_id]
    
    # get all fouls by team_id
    team_fouls_df = match_fouls_df[match_fouls_df['team_id'] == team_id]
    
    team_fouls_df['seconds'] = team_fouls_df['minute'] * 60 + team_fouls_df['second']

    # get all fouls before timestamp
    team_fouls_df = team_fouls_df[team_fouls_df['seconds'] <= seconds]
    
    # return count of fouls
    return len(team_fouls_df)

In [20]:
def make_advanced_fouls(fouls, goals, match_events, match_ids_info):
  """
  This function takes a DataFrame of fouls and a `match_ids_info` DataFrame and returns a new DataFrame with advanced foul metrics.

  Args:
    all_fouls: A DataFrame of fouls.
    match_ids_info: A DataFrame that contains information about the matches, such as the home team and the away team.

  Returns:
    A DataFrame of advanced fouls.
  """

  # Copy the original DataFrame.
  fouls_advanced = fouls.copy()

  fouls_advanced['seconds_till_now'] = fouls_advanced['minute'] * 60 + fouls_advanced['second']
  fouls_advanced['scoreline_till_now'] = fouls_advanced.apply(lambda x: get_scoreline(goals, match_events, x['match_id'], x['seconds_till_now']), axis=1)
  fouls_advanced['distance_to_goal'] = fouls_advanced.apply(lambda x: distance_to_goal(x['team'] == match_ids_info[x['match_id']][0], x['location']), axis=1)
  fouls_advanced['angle_to_goal'] = fouls_advanced.apply(lambda x: angle_to_goal(x['team'] == match_ids_info[x['match_id']][0], x['location']), axis=1)

  # Calculate the number of fouls committed by the player up to the time of the foul.
  # all_fouls_advanced['foul_count_player_till_now'] = all_fouls_advanced.apply(lambda x: previous_foul_count(all_fouls, x['match_id'], x['player_id'], x['seconds_till_now']), axis=1)

  # Calculate the number of fouls committed by the team up to the time of the foul.
  # all_fouls_advanced['foul_count_team_till_now'] = all_fouls_advanced.apply(lambda x: previous_foul_count_team(all_fouls, x['match_id'], x['team_id'], x['seconds_till_now']), axis=1)

  return fouls_advanced

In [21]:
all_fouls_advanced = make_advanced_fouls(all_fouls, all_goals, match_events, match_ids_info)
testing_fouls_advanced = make_advanced_fouls(testing_fouls, test_goals, match_events_test, match_ids_info_test)

In [22]:
all_fouls_advanced['foul_count_player_till_now'] = all_fouls_advanced.apply(lambda x: previous_foul_count(all_fouls, x['match_id'], x['player_id'], x['seconds_till_now']), axis=1)
all_fouls_advanced['foul_count_team_till_now'] = all_fouls_advanced.apply(lambda x: previous_foul_count_team(all_fouls, x['match_id'], x['team_id'], x['seconds_till_now']), axis=1)

testing_fouls_advanced['foul_count_player_till_now'] = testing_fouls_advanced.apply(lambda x: previous_foul_count(testing_fouls, x['match_id'], x['player_id'], x['seconds_till_now']), axis=1)
testing_fouls_advanced['foul_count_team_till_now'] = testing_fouls_advanced.apply(lambda x: previous_foul_count_team(testing_fouls, x['match_id'], x['team_id'], x['seconds_till_now']), axis=1)

all_fouls_advanced.to_pickle('all_fouls_advanced.pkl')
testing_fouls_advanced.to_pickle('testing_fouls_advanced.pkl')

In [23]:
len(all_fouls_advanced)

6807

In [24]:
# find count of each unique element in foul_committed_type in all_fouls_advanced
all_fouls_advanced['foul_committed_type'].value_counts()

Handball          904
Dangerous Play    253
Foul Out           36
Dive               32
6 Seconds           7
Backpass Pick       3
Name: foul_committed_type, dtype: int64

In [30]:
import numpy as np

type_dict = {'Handball': 0, 'Dangerous Play': 1, 'Foul Out': 2, 'Dive': 3, '6 Seconds': 4, 'Backpass Pick': 5}

card_dict = {'Yellow Card': 0, 'Second Yellow': 0, 'Red Card': 1}

def create_features(team_fouls_df):
    all_features = []
    all_labels = []

    for _, row in team_fouls_df.iterrows():
        # select features only where foul_committed_type is not in type_dict keys
        if row['foul_committed_type'] in type_dict.keys():
            continue
        
        team_making_foul = row['team']

        scoreline = row['scoreline_till_now']
        team_not_making_foul = [team for team in scoreline.keys() if team != team_making_foul][0]
        
        score_difference_till_now = scoreline[team_not_making_foul] - scoreline[team_making_foul]

        features = [row['minute'], score_difference_till_now, row['distance_to_goal'], row['angle_to_goal'], row['foul_count_player_till_now'], row['foul_count_team_till_now'], row['id']]

        all_features.append(features)
        all_labels.append([card_dict.get(row['foul_committed_card'], 2)])
    
    return all_features, all_labels

In [31]:
import pickle

all_the_features = []
all_the_labels = []

# use tqdm to iterate match_ids_list
for match_id in tqdm(match_ids_list):
    # get all fouls in the match
    match_fouls_df = all_fouls_advanced[all_fouls_advanced['match_id'] == match_id]

    feature, label = create_features(match_fouls_df)

    # append features and labels to all_the_features and all_the_labels
    all_the_features.extend(feature)
    all_the_labels.extend(label)

# save all_the_features and all_the_labels to files using pickle
with open('all_the_features.pkl', 'wb') as f:
    pickle.dump(all_the_features, f)

with open('all_the_labels.pkl', 'wb') as f:
    pickle.dump(all_the_labels, f)

100%|██████████| 623/623 [00:01<00:00, 389.88it/s]


In [32]:
all_the_features_test= []
all_the_labels_test = []

# use tqdm to iterate match_ids_list
for match_id in tqdm(match_ids_list_test):
    # get all fouls in the match
    match_fouls_df = testing_fouls_advanced[testing_fouls_advanced['match_id'] == match_id]

    feature, label = create_features(match_fouls_df)

    # append features and labels to all_the_features and all_the_labels
    all_the_features_test.extend(feature)
    all_the_labels_test.extend(label)

# save all_the_features and all_the_labels to files using pickle
with open('all_the_features_test.pkl', 'wb') as f:
    pickle.dump(all_the_features_test, f)

with open('all_the_labels_test.pkl', 'wb') as f:
    pickle.dump(all_the_labels_test, f)

100%|██████████| 128/128 [00:00<00:00, 542.94it/s]


In [ ]:
# # load all_the_features and all_the_labels from files using pickle
# with open('all_the_features.pkl', 'rb') as f:
#     all_the_features = pickle.load(f)

# with open('all_the_labels.pkl', 'rb') as f:
#     all_the_labels = pickle.load(f)



# # load all_the_features_test and all_the_labels_test from files using pickle
# with open('all_the_features_test.pkl', 'rb') as f:
#     all_the_features_test = pickle.load(f)

# with open('all_the_labels_test.pkl', 'rb') as f:
#     all_the_labels_test = pickle.load(f)

In [33]:
print (len(all_the_features))

# first 10 items in all_the_features and all_the_labels
print (all_the_features[:100])
print (all_the_labels[:100])

# print count of unique labels
print (np.unique(all_the_labels, return_counts=True))

5572
[[3, -1, 74.90367147209808, -0.2981063128391762, 1, 1, 'a40924e7-b5ca-4f23-9025-ac535d63dab8'], [8, -1, 82.67823171790747, 0.3557978590960649, 1, 2, '24f49483-d437-40c2-bf5f-c6b3e5456d48'], [54, -1, 29.04134983088768, 1.2773606430795648, 1, 4, '681f5c57-ed69-4bae-82e7-6d97233fb907'], [74, -1, 102.00705857929637, -0.011764163149758776, 1, 5, '7ea7ec5e-388e-4b20-8ab5-d2359cb92a65'], [55, -1, 77.53192890674138, -0.41979583169048146, 1, 2, '6238286d-a363-4ed3-abab-f322e0571e2d'], [81, -1, 81.17745992577004, 0.41459707844526655, 1, 4, '928a15d8-a10e-411c-953d-301663f2fada'], [92, -1, 84.38009243891595, -0.3787458155510841, 2, 5, 'd32eb3eb-ac10-4d1f-a2d6-1f7d1b1f035c'], [10, 0, 43.18564576337837, -0.09275632018814374, 1, 1, 'caaeb0e8-e589-4586-bf9c-56f0e75d1adf'], [11, 0, 70.61161377563892, -0.2140606835638215, 1, 2, '53666edb-be25-4515-ab85-23b572e06721'], [30, -1, 54.00925846556311, -0.018516402068009044, 1, 3, '214c40b2-a542-469e-9b9d-2b3ac936ccf8'], [41, -1, 49.72926703662542, 0.264

In [29]:
# save match_ids_list to file using pickle
with open('match_ids_list.pkl', 'wb') as f:
    pickle.dump(match_ids_list, f)

# save match_ids_list_test to file using pickle
with open('match_ids_list_test.pkl', 'wb') as f:
    pickle.dump(match_ids_list_test, f)